# 🚀 CRAFT Phase 3: Evaluate, Compare & Deploy

This notebook runs the final evaluation pipeline:

| Step | What it does |
|------|-------------|
| **1a** | Clone repo + authenticate HF |
| **1b** | Install dependencies |
| **2** | Discover ALL checkpoints (auto – 150,200,250,… up to final) |
| **3** | Evaluate BASE model (Phi-3-Mini) |
| **3b** | 🔍 Verification: test answer extraction with plain‑text format |
| **4** | Evaluate all discovered checkpoints (sweep) |
| **5** | 🏆 Champion selection – find the best checkpoint |
| **5b** | 📈 Performance visualization – accuracy & improvement curves |
| **6** | Merge + GGUF convert + quantize CHAMPION only |
| **7** | GGUF deployment verification |
| **8** | Generate benchmark report from REAL numbers |
| **9** | Upload to HuggingFace (gated on Samsung's bar) |
| **10** | Package delivery bundle |

---
## Step 1: Setup Environment

In [ ]:
# 1a. Clone the repository + authenticate Hugging Face
from kaggle_secrets import UserSecretsClient
import os
user_secrets = UserSecretsClient()
git_token = user_secrets.get_secret("GITHUB_PAT")

# Authenticate Hugging Face globally for dataset downloads
try:
    hf_token = user_secrets.get_secret("HF_TOKEN")
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        from huggingface_hub import login
        login(token=hf_token)
        print("✅ Hugging Face Token authenticated globally!")
except Exception:
    print("⚠️ HF_TOKEN not found in secrets. Downloads might be rate-limited.")

repo_url = f"https://{git_token}@github.com/VishalGastu30/CRAFT-SLM-Reasoning.git"
if not os.path.exists("CRAFT-SLM-Reasoning"):
    !git clone {repo_url}
os.chdir("CRAFT-SLM-Reasoning")
!git pull
print("✅ Repository ready.")

In [ ]:
# Install all dependencies including llama-cpp-python for GGUF inference
!pip install -q transformers bitsandbytes accelerate datasets peft trl \
    loguru pyyaml scipy numpy huggingface_hub sentence-transformers matplotlib

# Install llama-cpp-python with CUDA support for GGUF evaluation
# Comment this out and use the CPU version if CUDA install fails
!CMAKE_ARGS="-DLLAMA_CUDA=on" pip install -q llama-cpp-python \
    --upgrade --force-reinstall --no-cache-dir 2>/dev/null || \
    pip install -q llama-cpp-python --upgrade --force-reinstall --no-cache-dir

print("✅ All dependencies installed.")

---
## Step 2: Discover All Checkpoints
Scans `checkpoints/rl/` for any folder named `checkpoint-*` (e.g., 150, 200, 250, 300, 350, 400, 450, 500).

In [ ]:
import os, re, shutil

# Ensure the RL checkpoint directory exists
os.makedirs("checkpoints/rl", exist_ok=True)

# Auto-discover checkpoints from the working directory (or from /kaggle/input if attached)
def discover_checkpoints(base_dir="checkpoints/rl"):
    checkpoints = []
    if os.path.exists(base_dir):
        for item in os.listdir(base_dir):
            if item.startswith("checkpoint-") and os.path.isdir(os.path.join(base_dir, item)):
                # Extract step number
                match = re.search(r'checkpoint-(\d+)', item)
                if match:
                    step = int(match.group(1))
                    checkpoints.append((step, os.path.join(base_dir, item)))
    return sorted(checkpoints, key=lambda x: x[0])

# Also look in /kaggle/input in case checkpoints were attached as a dataset
input_checkpoints = []
for root, dirs, files in os.walk('/kaggle/input'):
    for d in dirs:
        if d.startswith("checkpoint-"):
            match = re.search(r'checkpoint-(\d+)', d)
            if match:
                step = int(match.group(1))
                src_path = os.path.join(root, d)
                target_path = f"checkpoints/rl/{d}"
                if not os.path.exists(target_path):
                    print(f"Copying {d} from {src_path} ...")
                    shutil.copytree(src_path, target_path)
                input_checkpoints.append((step, target_path))

# Combine and deduplicate (working directory takes precedence)
all_checkpoints = {}
for step, path in discover_checkpoints():
    all_checkpoints[step] = path
for step, path in input_checkpoints:
    if step not in all_checkpoints:
        all_checkpoints[step] = path

checkpoint_list = sorted(all_checkpoints.items())

if not checkpoint_list:
    print("❌ No checkpoints found in 'checkpoints/rl' or attached datasets.")
    print("   Please ensure you have run Phase 2 training and saved checkpoints.")
    # Exit gracefully
    import sys
    sys.exit(1)

print(f"✅ Discovered {len(checkpoint_list)} checkpoint(s):")
for step, path in checkpoint_list:
    print(f"   Step {step}: {path}")

---
## Step 3: Evaluate the Original Phi-3-Mini Baseline
Downloads the untrained base model and runs it on GSM8K, StrategyQA, and MMLU to get the "before" scores.

In [ ]:
import os
os.makedirs("craft_output", exist_ok=True)

# Evaluate the untrained base model — this is your "before" score
!python -m src.phase3_deploy.evaluator \
    --model-type hf \
    --model-path microsoft/Phi-3-mini-4k-instruct \
    --dataset all \
    --samples 100 \
    --label "Phi-3-Mini-Baseline" \
    --gpu \
    --output-json craft_output/baseline_results.json

import json
with open("craft_output/baseline_results.json") as f:
    bl = json.load(f)
print("\n📊 Baseline Results:")
print(f"  GSM8K:      {bl['gsm8k']['summary']['accuracy']:.1%}")
print(f"  StrategyQA: {bl['strategyqa']['summary']['accuracy']:.1%}")
print(f"  MMLU:       {bl['mmlu']['summary']['accuracy']:.1%}")
print("\n✅ Baseline evaluation complete.")

---
## Step 3b: 🔍 Verification of Answer Extraction (Plain Text Format)
Tests that the evaluator correctly extracts answers from the plain‑text format your model produces (`Step N:` + `Final Answer:`).

In [ ]:
from src.phase3_deploy.evaluator import BenchmarkEvaluator

eval_check = BenchmarkEvaluator(use_gpu=False)  # no GPU needed for this test

print("\n" + "="*60)
print("Verifying Answer Extraction (Plain Text Format)")
print("="*60)

# Test 1: GSM8K style
resp1 = "Step 1: 5 + 3 = 8\nStep 2: 8 - 2 = 6\nFinal Answer: 6"
pred1 = eval_check.extract_model_answer(resp1)
assert pred1 == "6", f"GSM8K extraction failed: got '{pred1}'"
print("✅ GSM8K: 'Final Answer: 6' →", pred1)

# Test 2: StrategyQA style
resp2 = "Step 1: Eiffel Tower built 1889.\nStep 2: US Civil War ended 1865.\nFinal Answer: no"
pred2 = eval_check.extract_model_answer(resp2)
assert pred2 == "no", f"StrategyQA extraction failed: got '{pred2}'"
print("✅ StrategyQA: 'Final Answer: no' →", pred2)

# Test 3: MMLU style
resp3 = "Step 1: Evaluate options.\nStep 2: B is correct.\nFinal Answer: B"
pred3 = eval_check.extract_model_answer(resp3)
assert pred3 == "B", f"MMLU extraction failed: got '{pred3}'"
print("✅ MMLU: 'Final Answer: B' →", pred3)

# Test 4: edge case – answer with extra text
resp4 = "Step 1: Compute 15% of 200 = 30.\nFinal Answer: 30 dollars"
pred4 = eval_check.extract_model_answer(resp4)
assert pred4 == "30", f"Numeric extraction with extra text failed: '{pred4}'"
print("✅ Numeric with extra text: '30 dollars' →", pred4)

print("\n" + "="*60)
print("✅ All extraction tests passed. Evaluator is ready for benchmarking.")
print("="*60 + "\n")

---
## Step 4: Evaluate All Discovered Checkpoints
Runs the evaluator on every checkpoint found in Step 2, storing results in `craft_output/results_step{step}.json`.

In [ ]:
import json, os, subprocess

checkpoint_results = {}

for step, ck_path in checkpoint_list:
    results_file = f"craft_output/results_step{step}.json"
    
    # Skip if already evaluated (optional, for resuming)
    if os.path.exists(results_file):
        print(f"\n⏩ Results for step {step} already exist – skipping evaluation.")
        with open(results_file) as f:
            checkpoint_results[step] = json.load(f)
        continue
    
    print(f"\n{'='*60}")
    print(f"Evaluating checkpoint step {step}...")
    print(f"{'='*60}")
    
    # Run evaluator
    !python -m src.phase3_deploy.evaluator \
        --model-type hf \
        --model-path {ck_path} \
        --dataset all \
        --samples 100 \
        --label "step-{step}" \
        --gpu \
        --output-json {results_file}
    
    if os.path.exists(results_file):
        with open(results_file) as f:
            res = json.load(f)
        checkpoint_results[step] = res
        gsm = res['gsm8k']['summary']['accuracy']
        strat = res['strategyqa']['summary']['accuracy']
        mmlu = res['mmlu']['summary']['accuracy']
        print(f"  ✅ Step {step}: GSM8K={gsm:.1%}, StrategyQA={strat:.1%}, MMLU={mmlu:.1%}")
    else:
        print(f"  ❌ Evaluation failed for step {step}")

---
## Step 5: 🏆 Champion Selection – The Arena
Compares all evaluated checkpoints and selects the one with the highest **average improvement** across GSM8K, StrategyQA, and MMLU. Also checks Samsung's requirements.

In [ ]:
import json, os

# Samsung's required minimum scores and delta
SAMSUNG_TARGETS = {
    "gsm8k":      {"min_score": 0.50, "min_delta": 0.05},
    "strategyqa": {"min_score": 0.65, "min_delta": 0.05},
    "mmlu":       {"min_score": 0.45, "min_delta": 0.05},
}

# Load baseline results
with open("craft_output/baseline_results.json") as f:
    baseline = json.load(f)

baseline_gsm = baseline['gsm8k']['summary']['accuracy']
baseline_strat = baseline['strategyqa']['summary']['accuracy']
baseline_mmlu = baseline['mmlu']['summary']['accuracy']

print(f"\n{'='*100}")
print(f"  CRAFT Checkpoint Sweep — The Arena")
print(f"{'='*100}")
print(f"\n  Samsung targets: GSM8K≥50% (+5%), StrategyQA≥65% (+5%), MMLU≥45% (+5%)")
print(f"\n{'Step':<8} {'GSM8K':>7} {'Δ GSM':>6} {'StratQA':>8} {'Δ Strat':>7} {'MMLU':>7} {'Δ MMLU':>6} {'Avg Δ':>7} {'Samsung':>9}")
print("-" * 100)

best_step = None
best_avg_delta = -999

for step in sorted(checkpoint_results.keys()):
    res = checkpoint_results[step]
    gsm = res['gsm8k']['summary']['accuracy']
    strat = res['strategyqa']['summary']['accuracy']
    mmlu = res['mmlu']['summary']['accuracy']
    d_gsm = gsm - baseline_gsm
    d_strat = strat - baseline_strat
    d_mmlu = mmlu - baseline_mmlu
    avg_d = (d_gsm + d_strat + d_mmlu) / 3
    
    # Samsung check
    passes_gsm = gsm >= 0.50 and d_gsm >= 0.05
    passes_strat = strat >= 0.65 and d_strat >= 0.05
    passes_mmlu = mmlu >= 0.45 and d_mmlu >= 0.05
    passes_count = sum([passes_gsm, passes_strat, passes_mmlu])
    samsung_label = "✅ PASS" if passes_count >= 2 else (f"⚠️ {passes_count}/3" if passes_count >= 1 else "❌ FAIL")
    
    print(f"{step:<8} {gsm:>6.1%} {d_gsm:>+5.1%} {strat:>7.1%} {d_strat:>+6.1%} {mmlu:>6.1%} {d_mmlu:>+5.1%} {avg_d:>+6.1%} {samsung_label:>9}")
    
    if avg_d > best_avg_delta:
        best_avg_delta = avg_d
        best_step = step

print("-" * 100)
print(f"\n  🏆 CHAMPION: checkpoint-{best_step} (avg improvement: {best_avg_delta:+.1%})")
print(f"{'='*100}\n")

# Save champion selection
champion_info = {
    "champion_step": best_step,
    "avg_improvement_percent": round(best_avg_delta * 100, 2),
    "baseline_gsm8k": round(baseline_gsm, 4),
    "baseline_strategyqa": round(baseline_strat, 4),
    "baseline_mmlu": round(baseline_mmlu, 4),
    "samsung_min_improvement_pct": 5.0,
    "passed": best_avg_delta >= 0.05,
}
with open("craft_output/champion_selection.json", "w") as f:
    json.dump(champion_info, f, indent=2)

CHAMPION_STEP = best_step
CHAMPION_PATH = f"checkpoints/rl/checkpoint-{best_step}"
print(f"Champion checkpoint path: {CHAMPION_PATH}")
print(f"Champion info saved to craft_output/champion_selection.json")

---
## Step 5b: 📈 Performance Visualization
Generates accuracy and improvement curves across all evaluated steps using `matplotlib`. Saves plots to `craft_output/performance_curves.png`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Prepare data
steps = sorted(checkpoint_results.keys())
gsm8k_acc = [checkpoint_results[s]['gsm8k']['summary']['accuracy'] for s in steps]
strategyqa_acc = [checkpoint_results[s]['strategyqa']['summary']['accuracy'] for s in steps]
mmlu_acc = [checkpoint_results[s]['mmlu']['summary']['accuracy'] for s in steps]

# Compute improvements
base_gsm, base_strat, base_mmlu = baseline_gsm, baseline_strat, baseline_mmlu
improvements = [(gsm8k_acc[i] - base_gsm, strategyqa_acc[i] - base_strat, mmlu_acc[i] - base_mmlu) for i in range(len(steps))]
avg_improvements = [(improvements[i][0] + improvements[i][1] + improvements[i][2])/3 for i in range(len(steps))]

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. GSM8K accuracy
ax1 = axes[0, 0]
ax1.plot(steps, gsm8k_acc, 'o-', label='GSM8K', color='#2E86C1', linewidth=2, markersize=8)
ax1.axhline(y=0.50, color='red', linestyle='--', linewidth=1.5, label='Samsung min 50%')
ax1.set_xlabel('Training Step')
ax1.set_ylabel('Accuracy')
ax1.set_title('GSM8K Performance')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. StrategyQA accuracy
ax2 = axes[0, 1]
ax2.plot(steps, strategyqa_acc, 's-', label='StrategyQA', color='#E67E22', linewidth=2, markersize=8)
ax2.axhline(y=0.65, color='red', linestyle='--', linewidth=1.5, label='Samsung min 65%')
ax2.set_xlabel('Training Step')
ax2.set_ylabel('Accuracy')
ax2.set_title('StrategyQA Performance')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. MMLU accuracy
ax3 = axes[1, 0]
ax3.plot(steps, mmlu_acc, 'd-', label='MMLU', color='#27AE60', linewidth=2, markersize=8)
ax3.axhline(y=0.45, color='red', linestyle='--', linewidth=1.5, label='Samsung min 45%')
ax3.set_xlabel('Training Step')
ax3.set_ylabel('Accuracy')
ax3.set_title('MMLU Performance')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Average improvement
ax4 = axes[1, 1]
ax4.plot(steps, avg_improvements, '^-', label='Avg Improvement (3 benchmarks)', color='#8E44AD', linewidth=2, markersize=8)
ax4.axhline(y=0.05, color='green', linestyle='--', linewidth=1.5, label='Samsung +5% target')
ax4.set_xlabel('Training Step')
ax4.set_ylabel('Improvement (percentage points)')
ax4.set_title('Average Improvement over Baseline')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('craft_output/performance_curves.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close('all')  # free memory
print("✅ Performance plots saved to craft_output/performance_curves.png")

---
## Step 6: Merge + GGUF Convert + Quantize (CHAMPION ONLY)
Merges the champion's LoRA weights, converts to GGUF, and quantizes to Q4_K_M for deployment.

In [ ]:
# Only quantize the champion — don't waste time on others
print(f"Merging and quantizing champion: checkpoint-{CHAMPION_STEP}")

# Disk cleanup utilities
import shutil, os, gc, torch

def print_disk_status():
    import subprocess
    result = subprocess.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True)
    print(result.stdout.strip())

print("Initial Disk Status:")
print_disk_status()

!python -m src.phase3_deploy.quantizer \
    --base-model microsoft/Phi-3-mini-4k-instruct \
    --lora-adapter {CHAMPION_PATH} \
    --merged-output craft_output/merged \
    --gguf-f16 craft_output/craft_phi3_f16.gguf \
    --gguf-quantized craft_output/craft_phi3_Q4_K_M.gguf

# Delete intermediate merged folder
if os.path.exists("craft_output/craft_phi3_f16.gguf") and os.path.exists("craft_output/merged"):
    shutil.rmtree("craft_output/merged")
    print("🗑️ Deleted merged HF folder (~7.5GB freed)")

assert os.path.exists("craft_output/craft_phi3_Q4_K_M.gguf"), \
    "GGUF not found! Check quantizer output above."

# Delete the FP16 GGUF after Q4 is confirmed
if os.path.exists("craft_output/craft_phi3_f16.gguf"):
    os.remove("craft_output/craft_phi3_f16.gguf")
    print("🗑️ Deleted F16 GGUF (~7.5GB freed)")

size_mb = os.path.getsize("craft_output/craft_phi3_Q4_K_M.gguf") / (1024*1024)
print(f"\n✅ Quantization complete! Size: {size_mb:.0f} MB")
print("Final Disk Status:")
print_disk_status()

---
## Step 7: GGUF Deployment Verification
Runs a quick 30-sample sanity check on the GGUF to make sure quantization didn't destroy quality.
The **official** Samsung score is from Step 4 (HF format).

In [ ]:
!python -m src.phase3_deploy.evaluator \
    --model-type gguf \
    --model-path craft_output/craft_phi3_Q4_K_M.gguf \
    --dataset all \
    --samples 30 \
    --label "CRAFT-GGUF-Q4" \
    --gpu \
    --output-json craft_output/gguf_verification_results.json

import json, os
with open("craft_output/gguf_verification_results.json") as f:
    gguf_res = json.load(f)

with open(f"craft_output/results_step{CHAMPION_STEP}.json") as f:
    hf_res = json.load(f)

print("\n📊 GGUF Quality Verification (30 samples):")
for ds in ["gsm8k", "strategyqa", "mmlu"]:
    hf_acc = hf_res[ds]['summary']['accuracy']
    gguf_acc = gguf_res[ds]['summary']['accuracy']
    delta = gguf_acc - hf_acc
    print(f"  {ds}: HF={hf_acc:.1%} → GGUF={gguf_acc:.1%} (delta={delta:+.1%})")
    if abs(delta) > 0.05:
        print(f"  ⚠️  GGUF degraded by more than 5%! Consider Q5_K_M quantization instead.")
    else:
        print(f"  ✅ GGUF quality retained.")

# Cleanup HF cache to free disk space
import shutil, gc, torch
torch.cuda.empty_cache()
gc.collect()
cache_path = os.path.expanduser("~/.cache/huggingface/hub")
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print("🗑️ Cleared HF model cache (~7.5GB freed)")

---
## Step 8: Generate Full Benchmark Report
Creates a professional Markdown report from **real** evaluation numbers. Nothing hardcoded.

In [ ]:
!python -m src.phase3_deploy.report_generator \
    --baseline-json craft_output/baseline_results.json \
    --champion-json craft_output/results_step{CHAMPION_STEP}.json \
    --champion-label "checkpoint-{CHAMPION_STEP}" \
    --ckpt150-json craft_output/results_step150.json \
    --ckpt200-json craft_output/results_step200.json \
    --ckpt250-json craft_output/results_step250.json \
    --report-md craft_output/benchmark_report.md

# Copy to docs/ for Samsung submission
!cp craft_output/benchmark_report.md docs/benchmark_results.md
print("✅ Report generated and copied to docs/benchmark_results.md")

---
## Step 9: Upload to HuggingFace (Gated on Samsung's Bar)
This cell will **only upload** if the champion checkpoint passes Samsung's minimum improvement requirements (≥2 benchmarks with +5% and absolute minimum).

In [ ]:
import json, os
from kaggle_secrets import UserSecretsClient

with open("craft_output/champion_selection.json") as f:
    champ = json.load(f)

champion_step = champ["champion_step"]
champion_json = f"craft_output/results_step{champion_step}.json"

with open(champion_json) as f:
    champion_res = json.load(f)
with open("craft_output/baseline_results.json") as f:
    baseline_res = json.load(f)

gsm_delta = champion_res['gsm8k']['summary']['accuracy'] - baseline_res['gsm8k']['summary']['accuracy']
strat_delta = champion_res['strategyqa']['summary']['accuracy'] - baseline_res['strategyqa']['summary']['accuracy']
mmlu_delta = champion_res['mmlu']['summary']['accuracy'] - baseline_res['mmlu']['summary']['accuracy']

gsm_abs = champion_res['gsm8k']['summary']['accuracy']
strat_abs = champion_res['strategyqa']['summary']['accuracy']
mmlu_abs = champion_res['mmlu']['summary']['accuracy']

passes_gsm = gsm_abs >= 0.50 and gsm_delta >= 0.05
passes_strat = strat_abs >= 0.65 and strat_delta >= 0.05
passes_mmlu = mmlu_abs >= 0.45 and mmlu_delta >= 0.05
passes_count = sum([passes_gsm, passes_strat, passes_mmlu])

print(f"\n📊 Samsung Gate Check:")
print(f"  GSM8K:      {gsm_abs:.1%} (need ≥50%) | Δ={gsm_delta:+.1%} (need ≥+5%) → {'✅' if passes_gsm else '❌'}")
print(f"  StrategyQA: {strat_abs:.1%} (need ≥65%) | Δ={strat_delta:+.1%} (need ≥+5%) → {'✅' if passes_strat else '❌'}")
print(f"  MMLU:       {mmlu_abs:.1%} (need ≥45%) | Δ={mmlu_delta:+.1%} (need ≥+5%) → {'✅' if passes_mmlu else '❌'}")
print(f"  Benchmarks passing: {passes_count}/3 (need ≥2)")

if passes_count < 2:
    print(f"\n🛑 UPLOAD BLOCKED. Model does not meet Samsung's minimum targets (need 2 passing).")
    print(f"   Consider using an earlier checkpoint or retraining.")
else:
    print(f"\n✅ Model passes Samsung's bar. Proceeding with upload...")
    
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    
    if not hf_token:
        print("⚠️  HF_TOKEN not in Kaggle Secrets. Cannot upload.")
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=hf_token)
        repo_id = "Aurvion/CRAFT-Phi3-Mini"
        api.create_repo(repo_id=repo_id, repo_type="model", private=False, exist_ok=True)
        
        # Upload GGUF model
        api.upload_file(
            path_or_fileobj="craft_output/craft_phi3_Q4_K_M.gguf",
            path_in_repo="craft_phi3_Q4_K_M.gguf",
            repo_id=repo_id
        )
        print(f"  ✅ GGUF model uploaded")
        
        # Upload benchmark report
        api.upload_file(
            path_or_fileobj="craft_output/benchmark_report.md",
            path_in_repo="benchmark_report.md",
            repo_id=repo_id
        )
        print(f"  ✅ Benchmark report uploaded")
        
        # Upload champion selection JSON (raw evidence)
        api.upload_file(
            path_or_fileobj="craft_output/champion_selection.json",
            path_in_repo="champion_selection.json",
            repo_id=repo_id
        )
        print(f"  ✅ Champion selection evidence uploaded")
        
        # Upload all evaluation result JSONs
        for step in checkpoint_results.keys():
            rp = f"craft_output/results_step{step}.json"
            if os.path.exists(rp):
                api.upload_file(
                    path_or_fileobj=rp,
                    path_in_repo=f"eval_results/step{step}.json",
                    repo_id=repo_id
                )
        print(f"  ✅ All evaluation JSONs uploaded")
        
        print(f"\n🚀 Upload complete: https://huggingface.co/{repo_id}")

---
## Step 10: Package Delivery Bundle
Creates the final tar.gz bundle for Samsung submission.

In [ ]:
!python -m src.phase3_deploy.packager \
    --gguf craft_output/craft_phi3_Q4_K_M.gguf \
    --dir craft_delivery \
    --archive craft_delivery_bundle.tar.gz

import os
size_mb = os.path.getsize("craft_delivery_bundle.tar.gz") / (1024*1024)
print(f"✅ Delivery bundle: craft_delivery_bundle.tar.gz ({size_mb:.0f} MB)")